In [ ]:
# ============================================================================
# 1. 설정 (Configuration)
# ============================================================================
import os
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 로드
env_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path.cwd() / '.env'
load_dotenv(env_path)

# 환경 변수에서 API 키 로드
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Pinecone 설정
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
INDEX_NAME = "target" 

XLSX_PATH = "data/output/target_filtered.xlsx"  # 상대 경로


In [23]:
# pinecone-client 제거 후 pinecone 설치
%pip uninstall pinecone-client -y
%pip install pinecone openai pandas openpyxl tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [24]:
# ============================================================================
# 2. 라이브러리 설치 및 임포트
# ============================================================================

# !pip install pinecone-client openai pandas openpyxl tqdm

from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time

# 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ 라이브러리 로드 완료")


✅ 라이브러리 로드 완료


In [27]:
# ============================================================================
# 3. 데이터 로드 및 전처리 (최적화 버전)
# ============================================================================

df = pd.read_excel(XLSX_PATH, engine='openpyxl')

print(f"전체 row 수: {len(df)}")
print(f"컬럼: {df.columns.tolist()}")

# 메타데이터로 사용할 컬럼들 정의
METADATA_COLUMNS = [
    'id', 'update_date', 'target_id', 'target_short_name', 'target_nation',
    'api_flag', 'application_date', 'application_number', 'abstract',
    'invention_name', 'ipc_code'
]

# 필요한 컬럼만 추출 (존재하는 컬럼만)
available_columns = [col for col in METADATA_COLUMNS if col in df.columns]
df_filtered = df[available_columns].copy()

print(f"사용 가능한 컬럼: {available_columns}")

# abstract 기준으로 결측치 제거 (임베딩에 필수)
df_filtered = df_filtered.dropna(subset=['abstract'])
df_filtered = df_filtered[df_filtered['abstract'].str.strip() != '']

# 중복 제거
df_filtered = df_filtered.drop_duplicates()

#
# ---------전체 데이터 사용 (테스트 시 아래 주석 해제)------------------
#TEST_LIMIT = 100
#df_filtered = df_filtered.head(TEST_LIMIT)
# ----------------------------------------------------------------

print(f"유효 데이터 수: {len(df_filtered)}")
df_filtered.head()

전체 row 수: 578096
컬럼: ['id', 'update_date', 'target_id', 'target_short_name', 'target_nation', 'api_flag', 'applicant', 'application_date', 'application_number', 'abstract', 'invention_name', 'ipc_code', 'claim']
사용 가능한 컬럼: ['id', 'update_date', 'target_id', 'target_short_name', 'target_nation', 'api_flag', 'application_date', 'application_number', 'abstract', 'invention_name', 'ipc_code']
유효 데이터 수: 578077


,id,update_date,target_id,target_short_name,target_nation,api_flag,application_date,application_number,abstract,invention_name,ipc_code
0,1,2025-11-22 20:57:49.000,4,+Automation Inc,Japan,KR,20160701,1020160083505,"본 발명은 직교좌표 로봇에 있어서, 구동수단; 상기 구동수단이 내장 또는 외장 설치...",직교좌표 로봇,B25J 9/02|B25J 9/10|B25J 19/00
1,2,2025-11-22 20:57:49.000,4,+Automation Inc,Japan,KR,20220725,1020220091898,스크류잭을 이용한 리프터에 대한 발명이 개시된다. 개시된 스크류잭을 이용한 리프터는...,스크류잭을 이용한 리프터,B66F 3/08
2,7,2025-11-22 20:57:49.000,4,+Automation Inc,Japan,KR,20221107,1020247016280,테스트 중인 디바이스와 관련된 센서 시스템을 교정하기 위한 센서 교정 시스템 및 이...,"차량 및 로봇의 라이다, 카메라, 레이더 및 초음파 센서의 자동화된 외적 교정을 위...",G01S 7/497|G01S 17/89|G01S 17/931|G01S 7/41|G0...
3,11,2025-11-22 20:57:49.000,4,+Automation Inc,Japan,KR,20161216,1020160172195,"본 발명은 일회용 식품 용기의 측정 정렬 시스템에 관한 것으로, 정렬 스테이지(16...",일회용 식품 용기의 측정 정렬 시스템,G01N 21/88|G06T 7/00
4,13,2025-11-22 20:57:49.000,4,+Automation Inc,Japan,KR,20231123,1020230164401,"본 발명은 높이 맞춤형 자동 패키징 시스템에 있어서, 박스를 이송하는 이송부; 및 ...",높이 맞춤형 자동 패키징 시스템,B65B 5/02|B65B 59/00|B65B 57/02|B31B 50/20|B31...


In [28]:
# ============================================================================
# 4. Pinecone 인덱스 생성 또는 연결
# ============================================================================

# OpenAI text-embedding-3-large: 3072차원
EMBEDDING_DIMENSION = 3072

# 인덱스 존재 여부 확인
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"인덱스 '{INDEX_NAME}' 생성 중...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION
        )
    )
    print(f"✅ 인덱스 '{INDEX_NAME}' 생성 완료")
else:
    print(f"✅ 기존 인덱스 '{INDEX_NAME}' 사용")

# 인덱스 연결
index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())


✅ 기존 인덱스 'target' 사용
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 25 Jan 2026 15:37:42 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-id': '7798743480534668524',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [32]:
# ============================================================================
# 5. 임베딩 생성 함수
# ============================================================================

def get_embedding(text: str, model: str = "text-embedding-3-large") -> list:
    """OpenAI 임베딩 생성"""
    text = text.replace("\n", " ").strip()
    if not text:
        return None
    
    response = openai_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding


def get_embeddings_batch(texts: list, model: str = "text-embedding-3-large") -> list:
    """배치로 임베딩 생성 (최대 2048개)"""
    # 빈 텍스트 전처리
    processed_texts = [t.replace("\n", " ").strip() if t else "" for t in texts]
    
    response = openai_client.embeddings.create(
        input=processed_texts,
        model=model
    )
    return [item.embedding for item in response.data]


print("✅ 임베딩 함수 정의 완료")


✅ 임베딩 함수 정의 완료


In [33]:
# ============================================================================
# 6. Pinecone에 데이터 업로드 (최적화 버전 - 배치 처리)
# ============================================================================

EMBEDDING_BATCH_SIZE = 200  # OpenAI 배치 크기 (Pinecone 4MB 제한으로 인해 축소)
UPSERT_BATCH_SIZE = 100     # Pinecone 업서트 배치 크기 (4MB 제한)

failed_count = 0
success_count = 0

total_records = len(df_filtered)
print(f"총 {total_records:,}개 데이터 업로드 시작...")
print(f"임베딩 배치: {EMBEDDING_BATCH_SIZE}개씩 / 업서트 배치: {UPSERT_BATCH_SIZE}개씩")
print("=" * 60)

# 시작 시간 기록
from datetime import datetime, timedelta
start_time = datetime.now()

def safe_str(value, max_length=None):
    """안전하게 문자열로 변환"""
    if pd.isna(value):
        return ""
    result = str(value)
    if max_length:
        return result[:max_length]
    return result

# 데이터프레임을 배치로 나눠서 처리
total_batches = (total_records + EMBEDDING_BATCH_SIZE - 1) // EMBEDDING_BATCH_SIZE

for batch_num, batch_start in enumerate(range(0, total_records, EMBEDDING_BATCH_SIZE), 1):
    batch_end = min(batch_start + EMBEDDING_BATCH_SIZE, len(df_filtered))
    batch_df = df_filtered.iloc[batch_start:batch_end]
    
    try:
        # 배치로 abstract 추출
        abstracts = batch_df['abstract'].tolist()
        
        # 배치로 임베딩 생성 (한 번의 API 호출로 여러 개 처리!)
        embeddings = get_embeddings_batch(abstracts)
        
        # 벡터 구성
        vectors_to_upsert = []
        for i, (idx, row) in enumerate(batch_df.iterrows()):
            metadata = {
                "id": safe_str(row.get('id', '')),
                "update_date": safe_str(row.get('update_date', '')),
                "target_id": safe_str(row.get('target_id', '')),
                "target_short_name": safe_str(row.get('target_short_name', '')),
                "target_nation": safe_str(row.get('target_nation', '')),
                "api_flag": safe_str(row.get('api_flag', '')),
                "application_date": safe_str(row.get('application_date', '')),
                "application_number": safe_str(row.get('application_number', '')),
                "abstract": safe_str(row.get('abstract', ''), max_length=1000),
                "invention_name": safe_str(row.get('invention_name', ''), max_length=500),
                "ipc_code": safe_str(row.get('ipc_code', ''))
            }
            
            vectors_to_upsert.append({
                "id": f"{metadata['target_short_name']}_{idx}",
                "values": embeddings[i],
                "metadata": metadata
            })
        
        # Pinecone에 배치 업서트
        for upsert_start in range(0, len(vectors_to_upsert), UPSERT_BATCH_SIZE):
            upsert_end = min(upsert_start + UPSERT_BATCH_SIZE, len(vectors_to_upsert))
            index.upsert(vectors=vectors_to_upsert[upsert_start:upsert_end])
        
        success_count += len(batch_df)
        
        # 진행 상황 출력
        elapsed = datetime.now() - start_time
        progress_pct = (success_count / total_records) * 100
        
        # 예상 남은 시간 계산
        if success_count > 0:
            estimated_total = elapsed * (total_records / success_count)
            remaining = estimated_total - elapsed
            remaining_str = str(remaining).split('.')[0]  # 초 단위까지만
        else:
            remaining_str = "계산 중..."
        
        print(f"[{batch_num}/{total_batches}] {success_count:,}/{total_records:,}개 완료 "
              f"({progress_pct:.1f}%) | 경과: {str(elapsed).split('.')[0]} | 남은 시간: {remaining_str}")
        
    except Exception as e:
        failed_count += len(batch_df)
        print(f"\n❌ 오류 (batch {batch_start}-{batch_end}): {e}")
        time.sleep(1)  # 에러 시 잠시 대기

total_elapsed = datetime.now() - start_time
print("\n" + "=" * 60)
print(f"✅ 업로드 완료!")
print(f"   성공: {success_count:,}개")
print(f"   실패: {failed_count:,}개")
print(f"   총 소요 시간: {str(total_elapsed).split('.')[0]}")
print("=" * 60)

총 578,077개 데이터 업로드 시작...
임베딩 배치: 200개씩 / 업서트 배치: 100개씩
[1/2891] 200/578,077개 완료 (0.0%) | 경과: 0:00:04 | 남은 시간: 3:15:26
[2/2891] 400/578,077개 완료 (0.1%) | 경과: 0:00:08 | 남은 시간: 3:16:19
[3/2891] 600/578,077개 완료 (0.1%) | 경과: 0:00:11 | 남은 시간: 3:09:56
[4/2891] 800/578,077개 완료 (0.1%) | 경과: 0:00:16 | 남은 시간: 3:12:31
[5/2891] 1,000/578,077개 완료 (0.2%) | 경과: 0:00:20 | 남은 시간: 3:14:20
[6/2891] 1,200/578,077개 완료 (0.2%) | 경과: 0:00:24 | 남은 시간: 3:13:56
[7/2891] 1,400/578,077개 완료 (0.2%) | 경과: 0:00:28 | 남은 시간: 3:12:48
[8/2891] 1,600/578,077개 완료 (0.3%) | 경과: 0:00:32 | 남은 시간: 3:16:00
[9/2891] 1,800/578,077개 완료 (0.3%) | 경과: 0:00:37 | 남은 시간: 3:19:19
[10/2891] 2,000/578,077개 완료 (0.3%) | 경과: 0:00:41 | 남은 시간: 3:20:42
[11/2891] 2,200/578,077개 완료 (0.4%) | 경과: 0:00:45 | 남은 시간: 3:20:34
[12/2891] 2,400/578,077개 완료 (0.4%) | 경과: 0:00:50 | 남은 시간: 3:21:35
[13/2891] 2,600/578,077개 완료 (0.4%) | 경과: 0:00:55 | 남은 시간: 3:23:35
[14/2891] 2,800/578,077개 완료 (0.5%) | 경과: 0:00:58 | 남은 시간: 3:21:23
[15/2891] 3,000/578,077개 완료 (0.5%) | 경

In [13]:
# ============================================================================
# 7. 업로드 결과 확인
# ============================================================================

stats = index.describe_index_stats()
print(f"인덱스 통계:")
print(f"  - 총 벡터 수: {stats.total_vector_count}")
print(f"  - 차원: {stats.dimension}")


인덱스 통계:
  - 총 벡터 수: 100
  - 차원: 3072


In [14]:
# ============================================================================
# 8. 검색 테스트 (유사 특허 검색)
# ============================================================================

def search_similar(query_text: str, top_k: int = 5):
    """쿼리 텍스트와 유사한 특허 검색"""
    query_embedding = get_embedding(query_text)
    
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    print(f"🔍 쿼리: '{query_text[:50]}...'\n")
    for match in results.matches:
        print(f"Score: {match.score:.4f}")
        print(f"  Target: {match.metadata['target_short_name']}")
        print(f"  Abstract: {match.metadata['abstract'][:100]}...")
        print()
    
    return results

# 테스트 검색
# search_similar("로봇 기술 관련 특허")
